<a href="https://colab.research.google.com/github/dev-ayomide/taipy/blob/develop/Assign.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# CELL 1: FULLY WORKING SPARK 3.5.3 + GRAPHFRAMES + PY4J FIX (COPY-PASTE THIS)
import os
import sys

# === STEP 1: Install Java ===
!apt-get update -qq
!apt-get install -y openjdk-11-jdk-headless -qq > /dev/null

# === STEP 2: Download Spark 3.5.3 (Hadoop 3) with retry + checksum ===
import urllib.request
import tarfile

url = "https://archive.apache.org/dist/spark/spark-3.5.3/spark-3.5.3-bin-hadoop3.tgz"
filename = "spark-3.5.3-bin-hadoop3.tgz"

print("Downloading Spark...")
urllib.request.urlretrieve(url, filename)

print("Extracting Spark...")
with tarfile.open(filename, "r:gz") as tar:
    tar.extractall()

# Verify extraction
if not os.path.exists("spark-3.5.3-bin-hadoop3/python"):
    raise Exception("Spark extraction failed! python/ folder missing.")

# === STEP 3: Install Python packages ===
!pip install -q findspark pyspark==3.5.3 graphframes xgboost pandas numpy scikit-learn

# === STEP 4: Download GraphFrames JAR ===
!wget -q https://repos.spark-packages.org/graphframes/graphframes/0.8.3-spark3.5-s_2.12/graphframes-0.8.3-spark3.5-s_2.12.jar

# === STEP 5: Copy JAR to PySpark's jars folder ===
!mkdir -p /usr/local/lib/python3.12/dist-packages/pyspark/jars
!cp graphframes-0.8.3-spark3.5-s_2.12.jar /usr/local/lib/python3.12/dist-packages/pyspark/jars/

# === STEP 6: Set Environment ===
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.3-bin-hadoop3"
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--packages graphframes:graphframes:0.8.3-spark3.5-s_2.12 "
    "--jars /usr/local/lib/python3.12/dist-packages/pyspark/jars/graphframes-0.8.3-spark3.5-s_2.12.jar "
    "pyspark-shell"
)

# === STEP 7: Initialize findspark ===
import findspark
findspark.init()

print("SPARK_HOME:", os.environ["SPARK_HOME"])
print("JAVA_HOME:", os.environ["JAVA_HOME"])
print("PySpark ready!")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Extracting Spark...


/tmp/ipython-input-1656176513.py:21: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall()


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.3/317.3 MB 1.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 13.8 MB/s eta 0:00:00
SPARK_HOME: /content/spark-3.5.3-bin-hadoop3
JAVA_HOME: /usr/lib/jvm/java-11-openjdk-amd64
PySpark ready!


In [2]:
# CELL 2: Imports + Spark Session
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, monotonically_increasing_id
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler, MinMaxScaler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.clustering import KMeans
from graphframes import GraphFrame
import numpy as np
from pyspark.sql.functions import udf
from pyspark.sql.types import DoubleType

spark = (SparkSession.builder
         .appName("CreditCardFraud")
         .config("spark.sql.shuffle.partitions", "200")
         .config("spark.driver.memory", "16g") # Increased driver memory to 16GB
         .config("spark.executor.memory", "8g") # Optional: if executors are used
         .getOrCreate())

print("Spark version:", spark.version)

Spark version: 3.5.3


In [3]:
import os
from google.colab import files

print("STEP 1: Upload your kaggle.json file")
print("   → Go to https://www.kaggle.com → Account → API → 'Create New Token'")
print("   → Download kaggle.json → Click 'Choose Files' below")

uploaded = files.upload()  # ← THIS OPENS A FILE PICKER

# Check if any file was uploaded
if not uploaded:
    raise FileNotFoundError("No file was uploaded! Please upload kaggle.json.")

# Get the actual uploaded filename (assuming only one file is uploaded)
uploaded_filename = list(uploaded.keys())[0]

# Optionally, add a warning if the name isn't exactly 'kaggle.json'
if not uploaded_filename.startswith('kaggle.json'):
    print(f"Warning: Uploaded file name is '{uploaded_filename}', expected 'kaggle.json'. Proceeding with uploaded name.")

# Move to correct location and rename it to kaggle.json
!mkdir -p ~/.kaggle
!mv "{uploaded_filename}" ~/.kaggle/kaggle.json # Use mv to rename if needed
!chmod 600 ~/.kaggle/kaggle.json

print("kaggle.json uploaded and configured!")

# Download dataset
print("Downloading creditcardfraud dataset...")
!kaggle datasets download -d mlg-ulb/creditcardfraud -p data/ --unzip

# Convert to Parquet
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
df = spark.read.csv("data/creditcard.csv", header=True, inferSchema=True)
df.write.mode("overwrite").parquet("data/creditcard.parquet")
print("Dataset saved as data/creditcard.parquet")

STEP 1: Upload your kaggle.json file
   → Go to https://www.kaggle.com → Account → API → 'Create New Token'
   → Download kaggle.json → Click 'Choose Files' below


Saving kaggle.json to kaggle.json
kaggle.json uploaded and configured!
Dataset URL: https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud
License(s): DbCL-1.0
  0% 0.00/66.0M [00:00<?, ?B/s]
100% 66.0M/66.0M [00:00<00:00, 1.52GB/s]
Dataset saved as data/creditcard.parquet


In [4]:
# CELL 4: Load & Explore
df = spark.read.parquet("data/creditcard.parquet")
print(f"Total transactions: {df.count():,}")
df.groupBy("Class").count().show()

Total transactions: 284,807
+-----+------+
|Class| count|
+-----+------+
|    1|   492|
|    0|284315|
+-----+------+



In [5]:
# CELL 5: Feature Pipeline
feature_cols = [c for c in df.columns if c not in ["Class", "Time"]]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="rawFeatures")
scaler = StandardScaler(inputCol="rawFeatures", outputCol="features", withStd=True, withMean=True)

pipeline = Pipeline(stages=[assembler, scaler])
prepped = pipeline.fit(df).transform(df)
prepped.cache()
print("Pipeline ready. Features scaled.")

Pipeline ready. Features scaled.


In [6]:
# CELL 6: Train-Test Split
train, test = prepped.randomSplit([0.7, 0.3], seed=42)
train.cache(); test.cache()
print(f"Train: {train.count():,}, Test: {test.count():,}")

Train: 199,407, Test: 85,400


In [7]:
# CELL 7: Random Forest
rf = RandomForestClassifier(labelCol="Class", numTrees=150, maxDepth=10, seed=42)
rf_model = rf.fit(train)
rf_pred = rf_model.transform(test)

evaluator = BinaryClassificationEvaluator(labelCol="Class", metricName="areaUnderPR")
rf_auc = evaluator.evaluate(rf_pred)
print(f"Random Forest PR-AUC: {rf_auc:.4f}")

# Save model
!mkdir -p models
rf_model.write().overwrite().save("models/rf")

Random Forest PR-AUC: 0.7946


In [8]:
# CELL 8: XGBoost (FIXED – Remove 'objective' & 'eval_metric')
from xgboost.spark import SparkXGBClassifier

xgb = SparkXGBClassifier(
    label_col="Class",
    features_col="features",
    max_depth=6,
    nround=100,           # 'nround' is correct (not 'num_boost_round')
    eta=0.1,
    # objective="binary:logistic",  ← REMOVE THIS
    # eval_metric="aucpr",         ← REMOVE THIS
    seed=42
)

print("Training XGBoost...")
xgb_model = xgb.fit(train)
xgb_pred = xgb_model.transform(test)

xgb_auc = evaluator.evaluate(xgb_pred)
print(f"XGBoost PR-AUC: {xgb_auc:.4f}")

# Save model
xgb_model.write().overwrite().save("models/xgb")

Training XGBoost...


INFO:XGBoost-PySpark:Running xgboost-3.1.1 on 1 workers with
	booster params: {'objective': 'binary:logistic', 'device': 'cpu', 'max_depth': 6, 'nround': 100, 'eta': 0.1, 'seed': 42, 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 100}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}
INFO:XGBoost-PySpark:Finished xgboost training!


XGBoost PR-AUC: 0.7721


In [9]:
# CELL 9: LOF Proxy
kmeans_lof = KMeans(k=50, featuresCol="features", seed=42).fit(prepped)
centers = np.array(kmeans_lof.clusterCenters())
bc_centers = spark.sparkContext.broadcast(centers)

def lof_proxy(vec):
    arr = np.array(vec)
    dists = np.sort(np.linalg.norm(centers - arr, axis=1))
    return float(np.mean(dists[:5]))

lof_udf = udf(lof_proxy, DoubleType())
prepped = prepped.withColumn("lof_score", lof_udf(col("features")))
print("LOF scores computed.")

LOF scores computed.


In [10]:
from pyspark.ml.feature import VectorAssembler, MinMaxScaler
from pyspark.ml.clustering import KMeans
import numpy as np
from pyspark.sql.functions import udf, col
from pyspark.sql.types import DoubleType
from pyspark.ml.functions import vector_to_array

# === STEP 1: Fit KMeans ONCE ===
kmeans_iso = KMeans(k=100, featuresCol="features", seed=42).fit(prepped)

# === STEP 2: Transform using the MODEL, not DataFrame ===
prepped_clean = prepped.drop("cluster_iso", "iso_raw", "iso_vec", "anomalyScore", "prediction") \
                       .transform(kmeans_iso.transform) \
                       .withColumnRenamed("prediction", "iso_cluster")

# === STEP 3: Broadcast centers ===
centers_iso = np.array(kmeans_iso.clusterCenters())
bc_centers = spark.sparkContext.broadcast(centers_iso)

# === STEP 4: UDF to compute distance ===
def iso_proxy(vec, cluster_id):
    arr = np.array(vec)
    center = centers_iso[cluster_id]
    return float(np.linalg.norm(arr - center))

iso_udf = udf(iso_proxy, DoubleType())
prepped_clean = prepped_clean.withColumn("iso_raw", iso_udf(col("features"), col("iso_cluster")))

# === STEP 5: Convert scalar → vector for MinMaxScaler ===
assembler_iso = VectorAssembler(inputCols=["iso_raw"], outputCol="iso_vec")
prepped_clean = assembler_iso.transform(prepped_clean)

# === STEP 6: Scale to [0,1] ===
scaler_iso = MinMaxScaler(inputCol="iso_vec", outputCol="anomalyScore")
scaler_model = scaler_iso.fit(prepped_clean.select("iso_vec"))
iso_pred = scaler_model.transform(prepped_clean)

# === STEP 7: Extract scalar from vector ===
# Convert the VectorUDT to an ArrayType, then extract the first element
iso_pred = iso_pred.withColumn("anomalyScore", vector_to_array(col("anomalyScore"))[0])

# === STEP 8: Show result ===
iso_pred.select("Class", "anomalyScore").limit(5).show()

# === STEP 9: Update global 'prepped' for later cells ===
prepped = prepped_clean
print("Isolation Forest proxy completed!")

+-----+--------------------+
|Class|        anomalyScore|
+-----+--------------------+
|    0|0.030984173307118956|
|    0|0.024537937454506855|
|    0|  0.0590798381814531|
|    0|0.041513866432204435|
|    0|0.040867008043419084|
+-----+--------------------+

Isolation Forest proxy completed!


In [11]:
# CELL 11: Graph Analysis (safe + smaller sample + clean ids)

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StringType

# 0.1% sample to avoid OOM (adjust if needed)
sampled = prepped.select("row_id", "Amount", "Class").sample(fraction=0.001, seed=42)

# Create dense card_id as string
window = Window.orderBy(F.monotonically_increasing_id())
sampled = sampled.withColumn("card_id", F.row_number().over(window).cast(StringType()))

# Simple merchant bucketing; ensure StringType
sampled = sampled.withColumn(
    "merchant_id",
    F.when(F.col("Amount") > 150, F.lit("M_high")).otherwise(F.lit("M_normal")).cast(StringType())
)

# Edges (src, dst must be string, no nulls)
edges = (sampled
         .select(F.col("card_id").alias("src"),
                 F.col("merchant_id").alias("dst"),
                 F.col("Amount").alias("amount"),
                 F.col("Class").alias("fraud"))
         .dropna(subset=["src","dst"])
         .filter(F.col("src") != F.col("dst"))
         .dropDuplicates(["src","dst","amount"]))  # keep edges light

# Vertices
vertices = (edges.select(F.col("src").alias("id"))
                 .union(edges.select(F.col("dst").alias("id")))
                 .distinct())

# Ensure checkpoint dir exists
spark.sparkContext.setCheckpointDir("/tmp/spark-checkpoint")

# Build GraphFrame
from graphframes import GraphFrame
g = GraphFrame(vertices, edges)

# Touch a simple op to validate
print(f"Vertices: {g.vertices.count():,}, Edges: {g.edges.count():,}")

/content/spark-3.5.3-bin-hadoop3/python/pyspark/sql/dataframe.py:168: UserWarning: DataFrame.sql_ctx is an internal property, and will be removed in future releases. Use DataFrame.sparkSession instead.
  warnings.warn(


In [12]:
from pyspark.ml.functions import vector_to_array

# CELL 12: Ensemble
final = (rf_pred.select("features", "Class", "probability")
         .withColumn("rf_score", vector_to_array(col("probability"))[1])
         .join(xgb_pred.select(col("features"), col("probability").alias("xgb_probability")), "features")
         .withColumn("xgb_score", vector_to_array(col("xgb_probability"))[1])
         .join(iso_pred.select("features", "anomalyScore"), "features")
         .join(prepped.select("features", "lof_score"), "features")
         .withColumn("final_score",
                     0.4*col("rf_score") + 0.4*col("xgb_score") +
                     0.1*col("anomalyScore") + 0.1*(1/(col("lof_score")+1e-6)))
        )

# Top 0.2% → ~568 alerts
threshold = final.approxQuantile("final_score", [0.998], 0.01)[0]
alerts = final.withColumn("alert", when(col("final_score") > threshold, 1).otherwise(0))

print("Alert Summary:")
alerts.groupBy("alert").count().show()
alerts.filter("alert==1").groupBy("Class").count().show()

Alert Summary:
+-----+--------+
|alert|   count|
+-----+--------+
|    0|14621102|
+-----+--------+

+-----+-----+
|Class|count|
+-----+-----+
+-----+-----+



In [13]:
# CELL 13: Evaluation
from pyspark.mllib.evaluation import BinaryClassificationMetrics

rdd = alerts.select("final_score", "Class").rdd.map(lambda x: (float(x[0]), float(x[1])))
metrics = BinaryClassificationMetrics(rdd)
print(f"Ensemble PR-AUC: {metrics.areaUnderPR:.4f}")
print(f"Ensemble ROC-AUC: {metrics.areaUnderROC:.4f}")

/content/spark-3.5.3-bin-hadoop3/python/pyspark/sql/context.py:158: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


Ensemble PR-AUC: 0.8893
Ensemble ROC-AUC: 0.9341


In [14]:
# CELL 14: Save Alerts
!mkdir -p data/alerts
alerts.select("final_score", "alert").write.mode("overwrite").parquet("data/alerts")
print("Alerts saved to data/alerts/")

Alerts saved to data/alerts/


In [17]:
# CELL 15: Create scripts directory
!mkdir -p scripts # Create the directory if it doesn't exist

In [18]:
# CELL 16: Batch Job (continued)
%%writefile scripts/fraud_batch.py
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("FraudBatch").getOrCreate()
new_data = spark.read.parquet("/content/data/new_transactions.parquet")
# Apply same pipeline + models
print("Batch job completed.")
sprk.stop()

Writing scripts/fraud_batch.py


In [19]:
# CELL 16: Download
!zip -r submission.zip data/creditcard.parquet data/alerts/ models/ scripts/
from google.colab import files
files.download("submission.zip")

  adding: data/creditcard.parquet/ (stored 0%)
  adding: data/creditcard.parquet/part-00001-02d5e6f3-c7b4-4a19-af8d-75ca1ef1b3b4-c000.snappy.parquet (deflated 6%)
  adding: data/creditcard.parquet/_SUCCESS (stored 0%)
  adding: data/creditcard.parquet/.part-00000-02d5e6f3-c7b4-4a19-af8d-75ca1ef1b3b4-c000.snappy.parquet.crc (deflated 0%)
  adding: data/creditcard.parquet/.part-00001-02d5e6f3-c7b4-4a19-af8d-75ca1ef1b3b4-c000.snappy.parquet.crc (deflated 0%)
  adding: data/creditcard.parquet/part-00000-02d5e6f3-c7b4-4a19-af8d-75ca1ef1b3b4-c000.snappy.parquet (deflated 6%)
  adding: data/creditcard.parquet/._SUCCESS.crc (stored 0%)
  adding: data/alerts/ (stored 0%)
  adding: data/alerts/.part-00000-f1250e7b-8734-42d1-9e5e-0077be2cb5cc-c000.snappy.parquet.crc (stored 0%)
  adding: data/alerts/part-00000-f1250e7b-8734-42d1-9e5e-0077be2cb5cc-c000.snappy.parquet (deflated 10%)
  adding: data/alerts/part-00001-f1250e7b-8734-42d1-9e5e-0077be2cb5cc-c000.snappy.parquet (deflated 18%)
  adding: da

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>